# GNN Embedding Pipeline

Five phases in one notebook:

1. **Node features** — embed 1500 papers with Nomic → `X.npy`
2. **kNN graph** — build semantic similarity graph → `edge_index.pt`
3. **GNN training** — 2-layer GraphSAGE with link prediction → `gnn_weights.pt`
4. **Final embeddings** — GNN forward pass + ablation vs raw Nomic → `Z.npy`
5. **Benchmark submission** — 100 queries → `results/andrew_gnn_v1.jsonl`

**Prerequisites**: run `ss-load-dataset.ipynb` first to produce `papers.parquet`.

**Install**: `pip install sentence-transformers torch torch-geometric pandas numpy tqdm`

---
## Phase 1 — Node Features

Each paper is embedded as `title + abstract` using `nomic-embed-text-v1.5` with the
`search_document:` prefix — identical to Massimo's abstract embedding step.
Nomic weights are frozen for the entire pipeline.

Output: `X.npy` of shape `(N, 768)` where rows correspond to `papers.parquet` idx.

In [1]:
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"

papers = pd.read_parquet("papers.parquet")
print(f"Loaded {len(papers)} papers")

def clean_text(t: str) -> str:
    return re.sub(r"\s+", " ", str(t)).strip()

# Concatenate title + abstract; use search_document prefix for Nomic
texts = [
    "search_document: " + clean_text(row["title"]) + " " + clean_text(row["abstract"])
    for _, row in papers.iterrows()
]

print(f"Example text: {texts[0][:120]}...")

c:\Users\andre\OneDrive\Desktop\projects\fml-witty-pandas\papers\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 1500 papers
Example text: search_document: C-Terminal Amino Acids 471-507 of Avian Hepatitis E Virus Capsid Protein Are Crucial for Binding to Avi...


In [2]:
embed_model = SentenceTransformer(MODEL_NAME, trust_remote_code=True)

X = embed_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
)
X = np.array(X, dtype=np.float32)

np.save("X.npy", X)
print(f"Saved X.npy — shape: {X.shape}")

c:\Users\andre\OneDrive\Desktop\projects\fml-witty-pandas\papers\venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\andre\.cache\huggingface\hub\models--nomic-ai--nomic-embed-text-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
<All keys matched successfully>
Batches: 100%|██████████| 47/47 [00:

Saved X.npy — shape: (1500, 768)


---
## Phase 2 — kNN Graph Construction

Because the 1500 papers are randomly sampled across all fields, expected citation edges
within this corpus ≈ 0. A semantic kNN graph is the only viable structure.

Steps:
- Compute full cosine similarity matrix over `X` (dot product; vectors already L2-normalised)
- For each node take its top-K most similar neighbors (excluding self)
- Symmetrise so edges are undirected
- Deduplicate

**K sensitivity**: K=7 is the default. Rerun Phase 2 + 3 + 4 at K=5 and K=10 for ablation.
Rule of thumb: if mean pairwise cosine sim in Z is > 1.5× that of X, K is too high.

In [3]:
import torch
import numpy as np

K = 7  # neighbors per node — tune this for ablation

X = np.load("X.npy")
N = X.shape[0]

# Full cosine similarity matrix — shape (N, N)
# X is already L2-normalised so dot product = cosine similarity
sim = X @ X.T

# Zero out self-similarity so each node doesn't connect to itself
np.fill_diagonal(sim, -1.0)

# Top-K neighbors per node
topk_indices = np.argsort(sim, axis=1)[:, -K:]  # shape (N, K)

# Build directed edge list
src_list = []
dst_list = []
for i in range(N):
    for j in topk_indices[i]:
        src_list.append(i)
        dst_list.append(int(j))

# Symmetrise: add reverse edges
src_t = torch.tensor(src_list + dst_list, dtype=torch.long)
dst_t = torch.tensor(dst_list + src_list, dtype=torch.long)
edge_index = torch.stack([src_t, dst_t], dim=0)

# Deduplicate
edge_index = torch.unique(edge_index, dim=1)

torch.save(edge_index, "edge_index.pt")

num_edges = edge_index.shape[1]
avg_degree = num_edges / N

print(f"K             : {K}")
print(f"Nodes         : {N}")
print(f"Edges (directed, deduped): {num_edges}")
print(f"Avg degree    : {avg_degree:.1f}")
print(f"Saved edge_index.pt")

K             : 7
Nodes         : 1500
Edges (directed, deduped): 15442
Avg degree    : 10.3
Saved edge_index.pt


In [4]:
# Connected-components check — expect 1 component at K >= 5 for 1500 nodes
import torch

edge_index = torch.load("edge_index.pt")
N = int(edge_index.max().item()) + 1

# Union-Find
parent = list(range(N))

def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    parent[find(a)] = find(b)

for s, d in zip(edge_index[0].tolist(), edge_index[1].tolist()):
    union(s, d)

components = len(set(find(i) for i in range(N)))
print(f"Connected components: {components}")
if components > 1:
    print(f"WARNING: graph has {components} components — consider increasing K")
else:
    print("OK: graph is fully connected")

Connected components: 1
OK: graph is fully connected


---
## Phase 3 — GNN Model + Training

**Architecture**: 2-layer GraphSAGE with mean aggregation
- Layer 1: 768 → 512 with ReLU + dropout(0.3)
- Layer 2: 512 → 768 with L2 normalisation on output

**Training objective**: link prediction (self-supervised, no labels needed)
- Positive pairs: edges in `edge_index`
- Negative pairs: randomly sampled non-edges (NEG_RATIO × more negatives than positives)
- Loss: binary cross-entropy on dot(u, v) for each pair
- The GNN learns to push connected papers together and disconnected papers apart

Trains in seconds on CPU for 1500 nodes.

In [5]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
import numpy as np

class PaperGNN(torch.nn.Module):
    def __init__(self, in_dim: int = 768, hidden_dim: int = 512, out_dim: int = 768):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, out_dim)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.conv2(x, edge_index)
        return F.normalize(x, dim=-1)  # L2-normalise for cosine retrieval

In [6]:
# -------------------------------------------------------
# Training setup
# -------------------------------------------------------

EPOCHS    = 100
LR        = 1e-3
NEG_RATIO = 5   # negative edges per positive edge

X          = np.load("X.npy")
edge_index = torch.load("edge_index.pt")
x          = torch.tensor(X, dtype=torch.float)
N          = x.shape[0]
num_edges  = edge_index.shape[1]

gnn       = PaperGNN()
optimizer = torch.optim.Adam(gnn.parameters(), lr=LR)

def sample_negatives(N: int, num_neg: int) -> tuple[torch.Tensor, torch.Tensor]:
    """Sample random non-self-loop node pairs as negative edges."""
    src = torch.randint(0, N, (num_neg,))
    dst = torch.randint(0, N, (num_neg,))
    mask = src != dst
    return src[mask], dst[mask]

print(f"Nodes: {N}  |  Edges: {num_edges}  |  Epochs: {EPOCHS}")
print("Training...")

Nodes: 1500  |  Edges: 15442  |  Epochs: 100
Training...


In [7]:
# -------------------------------------------------------
# Training loop
# -------------------------------------------------------

gnn.train()
for epoch in range(1, EPOCHS + 1):
    optimizer.zero_grad()

    z = gnn(x, edge_index)

    # Positive scores: dot product of each connected pair
    pos_src   = edge_index[0]
    pos_dst   = edge_index[1]
    pos_score = (z[pos_src] * z[pos_dst]).sum(dim=1)

    # Negative scores: random non-connected pairs
    neg_src, neg_dst = sample_negatives(N, num_edges * NEG_RATIO)
    neg_score = (z[neg_src] * z[neg_dst]).sum(dim=1)

    pos_loss = F.binary_cross_entropy_with_logits(
        pos_score, torch.ones_like(pos_score)
    )
    neg_loss = F.binary_cross_entropy_with_logits(
        neg_score, torch.zeros_like(neg_score)
    )
    loss = pos_loss + neg_loss

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch:3d} | loss: {loss.item():.4f} "
              f"(pos: {pos_loss.item():.4f}, neg: {neg_loss.item():.4f})")

torch.save(gnn.state_dict(), "gnn_weights.pt")
print("\nSaved gnn_weights.pt")

Epoch  10 | loss: 1.1401 (pos: 0.3986, neg: 0.7415)
Epoch  20 | loss: 1.0958 (pos: 0.3651, neg: 0.7307)
Epoch  30 | loss: 1.0864 (pos: 0.3609, neg: 0.7254)
Epoch  40 | loss: 1.0818 (pos: 0.3585, neg: 0.7233)
Epoch  50 | loss: 1.0777 (pos: 0.3574, neg: 0.7203)
Epoch  60 | loss: 1.0761 (pos: 0.3569, neg: 0.7192)
Epoch  70 | loss: 1.0737 (pos: 0.3559, neg: 0.7178)
Epoch  80 | loss: 1.0735 (pos: 0.3553, neg: 0.7183)
Epoch  90 | loss: 1.0717 (pos: 0.3551, neg: 0.7166)
Epoch 100 | loss: 1.0704 (pos: 0.3548, neg: 0.7156)

Saved gnn_weights.pt


---
## Phase 4 — Final Embeddings + Ablation

Run the trained GNN in eval mode to produce `Z.npy` — the neighbourhood-aware embeddings
used for retrieval.

Then two checks:

1. **Over-smoothing check**: compare mean pairwise cosine similarity in X vs Z.
   If Z similarity is > 1.5× X similarity, K is too high — reduce it and retrain.

2. **Retrieval ablation**: for 5 sample queries, show top-5 results from raw Nomic (X)
   vs GNN (Z) side by side to make the difference visible.

In [8]:
import torch
import numpy as np

X          = np.load("X.npy")
edge_index = torch.load("edge_index.pt")
x          = torch.tensor(X, dtype=torch.float)

gnn = PaperGNN()
gnn.load_state_dict(torch.load("gnn_weights.pt"))
gnn.eval()

with torch.no_grad():
    Z = gnn(x, edge_index).numpy().astype(np.float32)

np.save("Z.npy", Z)
print(f"Saved Z.npy — shape: {Z.shape}")

Saved Z.npy — shape: (1500, 768)


In [9]:
# -------------------------------------------------------
# Over-smoothing check
# If mean_sim_Z >> mean_sim_X the GNN has collapsed
# embeddings toward a common mean — reduce K and retrain
# -------------------------------------------------------

X = np.load("X.npy")
Z = np.load("Z.npy")

# Sample 200 random pairs to estimate mean pairwise cosine similarity
rng   = np.random.default_rng(42)
N     = X.shape[0]
pairs = rng.integers(0, N, size=(2000, 2))
pairs = pairs[pairs[:, 0] != pairs[:, 1]][:1000]

sim_X = float(np.mean([(X[a] @ X[b]) for a, b in pairs]))
sim_Z = float(np.mean([(Z[a] @ Z[b]) for a, b in pairs]))

print(f"Mean pairwise cosine sim — X (raw Nomic): {sim_X:.4f}")
print(f"Mean pairwise cosine sim — Z (GNN)      : {sim_Z:.4f}")
print(f"Ratio Z/X                               : {sim_Z / sim_X:.2f}x")

if sim_Z > sim_X * 1.5:
    print("\nWARNING: over-smoothing detected — consider reducing K in Phase 2 and retraining")
else:
    print("\nOK: embeddings remain well-separated")

Mean pairwise cosine sim — X (raw Nomic): 0.5439
Mean pairwise cosine sim — Z (GNN)      : -0.0001
Ratio Z/X                               : -0.00x

OK: embeddings remain well-separated


In [10]:
# -------------------------------------------------------
# Retrieval ablation — raw Nomic X vs GNN Z
# -------------------------------------------------------

import pandas as pd
from sentence_transformers import SentenceTransformer

papers      = pd.read_parquet("papers.parquet")
embed_model = SentenceTransformer(MODEL_NAME, trust_remote_code=True)

SAMPLE_QUERIES = [
    "graph neural networks for molecules",
    "machine learning for healthcare",
    "federated learning privacy",
    "transformer attention mechanism",
    "contrastive learning representations",
]

for query in SAMPLE_QUERIES:
    q_emb = embed_model.encode(
        f"search_query: {query}",
        normalize_embeddings=True,
        show_progress_bar=False,
    )

    scores_X = X @ q_emb
    scores_Z = Z @ q_emb

    top5_X = np.argsort(scores_X)[::-1][:5]
    top5_Z = np.argsort(scores_Z)[::-1][:5]

    print(f"\n{'='*70}")
    print(f"Query: '{query}'")
    print(f"{'='*70}")
    print("  Raw Nomic (X):")
    for rank, idx in enumerate(top5_X, 1):
        title = papers.iloc[idx]["title"][:75]
        print(f"    {rank}. [{scores_X[idx]:.3f}] {title}")
    print("  GNN (Z):")
    for rank, idx in enumerate(top5_Z, 1):
        title = papers.iloc[idx]["title"][:75]
        print(f"    {rank}. [{scores_Z[idx]:.3f}] {title}")

<All keys matched successfully>



Query: 'graph neural networks for molecules'
  Raw Nomic (X):
    1. [0.728] AntiBMPNN: Structure‐Guided Graph Neural Networks for Precision Antibody En
    2. [0.705] Geometry-Aware Protein-Protein Binding Site Prediction Using Geometric Lear
    3. [0.696] Brain-inspired Hierarchical Attention Recurrent CNN for Image Classificatio
    4. [0.685] TinyMo: Graph-Level Memory Optimizer for Tiny Machine Learning
    5. [0.666] Improved Model Based Deep Learning Using Monotone Operator Learning (Mol)
  GNN (Z):
    1. [0.062] Strength of Preference and Decision Making Under Risk
    2. [0.058] N400 differences between physical and mental metaphors: The role of Theorie
    3. [0.058] Are concepts a natural kind? On concept eliminativism
    4. [0.057] A step into the anarchist's mind: examining political attitudes and ideolog
    5. [0.054] The association of childhood socio‐economic position with later life brain 

Query: 'machine learning for healthcare'
  Raw Nomic (X):
    1. [0.722] L

---
## Phase 5 — Benchmark Submission

Generates `results/andrew_gnn_v1.jsonl` in the shared evaluation format.

For each of the 100 benchmark queries:
1. Encode query with the same `search_query:` Nomic prefix
2. Cosine score against `Z` (dot product; both are L2-normalised)
3. Apply year filter if present (same logic as Massimo)
4. Deduplicate by `paperId`
5. Take top-10

Validated against the same rules as `massimo_weighted_semantic_v1.jsonl`.

In [11]:
import json
import re
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer

Z           = np.load("Z.npy")
papers      = pd.read_parquet("papers.parquet")
embed_model = SentenceTransformer(MODEL_NAME, trust_remote_code=True)

BENCHMARK_PATH   = Path("../evaluation layer/benchmark_queries.jsonl")
RUN_ID           = "andrew_gnn_v1"
SUBMISSION_PATH  = Path(f"results/{RUN_ID}.jsonl")

benchmark_queries = []
with open(BENCHMARK_PATH, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            benchmark_queries.append(json.loads(line))

print(f"Loaded {len(benchmark_queries)} benchmark queries")
print("Sample:", benchmark_queries[0])

<All keys matched successfully>


Loaded 100 benchmark queries
Sample: {'queryId': 'q_001', 'query': 'machine learning for healthcare', 'year': None, 'fields': 'paperId,title,abstract,authors,year,publicationDate,fieldsOfStudy,s2FieldsOfStudy,url'}


In [12]:
# -------------------------------------------------------
# Year-filter helpers — identical to Massimo's implementation
# -------------------------------------------------------

def parse_year_filter(year_value):
    if year_value is None:
        return None
    y = str(year_value).strip()
    if not y or y.lower() == "null":
        return None
    if re.fullmatch(r"\d{4}", y):
        return ("exact", int(y))
    m = re.fullmatch(r"(\d{4})-", y)
    if m:
        return ("min", int(m.group(1)))
    m = re.fullmatch(r"(\d{4})-(\d{4})", y)
    if m:
        return ("range", int(m.group(1)), int(m.group(2)))
    return None


def year_matches_filter(paper_year, year_filter) -> bool:
    if year_filter is None:
        return True
    try:
        py = int(float(paper_year))
    except Exception:
        return False
    kind = year_filter[0]
    if kind == "exact":
        return py == year_filter[1]
    if kind == "min":
        return py >= year_filter[1]
    if kind == "range":
        return year_filter[1] <= py <= year_filter[2]
    return True

In [13]:
# -------------------------------------------------------
# Retrieval function
# -------------------------------------------------------

def retrieve_top_k(query_text: str, year_value, k: int = 10) -> list[dict]:
    q_emb = embed_model.encode(
        f"search_query: {query_text}",
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    year_filter = parse_year_filter(year_value)

    scores = Z @ q_emb  # shape (N,), cosine similarity

    results = []
    for idx in range(len(papers)):
        row = papers.iloc[idx]
        if not year_matches_filter(row.get("year"), year_filter):
            continue
        results.append({"paperId": row["paperId"], "score": float(scores[idx])})

    results.sort(key=lambda r: r["score"], reverse=True)

    # deduplicate by paperId
    deduped = []
    seen: set[str] = set()
    for r in results:
        if r["paperId"] not in seen:
            deduped.append(r)
            seen.add(r["paperId"])
        if len(deduped) >= k:
            break

    return deduped

In [14]:
# -------------------------------------------------------
# Build submission rows
# -------------------------------------------------------

submission_rows = []

for q in benchmark_queries:
    query_id   = q["queryId"]
    query_text = q["query"]
    year_value = q.get("year", None)

    top_results = retrieve_top_k(query_text, year_value, k=10)

    submission_rows.append({
        "runId":   RUN_ID,
        "queryId": query_id,
        "results": [
            {"rank": rank, "paperId": r["paperId"], "score": r["score"]}
            for rank, r in enumerate(top_results, start=1)
        ],
    })

print(f"Built {len(submission_rows)} submission rows")

Built 100 submission rows


In [15]:
# -------------------------------------------------------
# Validate — same rules as Massimo
# -------------------------------------------------------

for row in submission_rows:
    assert "runId"   in row
    assert "queryId" in row
    assert "results" in row

    assert len(row["results"]) == 10, \
        f"{row['queryId']} has {len(row['results'])} results, expected 10"

    ranks = [r["rank"] for r in row["results"]]
    assert ranks == list(range(1, 11)), \
        f"{row['queryId']} has bad ranks: {ranks}"

    pids = [r["paperId"] for r in row["results"]]
    assert all(isinstance(p, str) and p for p in pids), \
        f"{row['queryId']} has invalid paperIds"
    assert len(set(pids)) == 10, \
        f"{row['queryId']} has duplicate paperIds"

print("Validation passed")

Validation passed


In [16]:
# -------------------------------------------------------
# Write JSONL submission
# -------------------------------------------------------

SUBMISSION_PATH.parent.mkdir(exist_ok=True)
with open(SUBMISSION_PATH, "w", encoding="utf-8") as f:
    for row in submission_rows:
        f.write(json.dumps(row) + "\n")

print(f"Wrote {SUBMISSION_PATH}")
print(f"Sample row: {submission_rows[0]}")

Wrote results\andrew_gnn_v1.jsonl
Sample row: {'runId': 'andrew_gnn_v1', 'queryId': 'q_001', 'results': [{'rank': 1, 'paperId': '6a57fab7e3ddd40c7b5f11ed47d9f662c02ff7fb', 'score': 0.043231766670942307}, {'rank': 2, 'paperId': 'd3300b70bae2013651190a4c3655f35cf30546a2', 'score': 0.043066687881946564}, {'rank': 3, 'paperId': '559cbdd2a3c93fb9b11d519256d8794aadf20f32', 'score': 0.042703863233327866}, {'rank': 4, 'paperId': '0ca327e9dd04b85560cbc02fd6be08094274110a', 'score': 0.04188890755176544}, {'rank': 5, 'paperId': 'bebac30e1104b895a472d27c218243065393a0b7', 'score': 0.041799288243055344}, {'rank': 6, 'paperId': '4ebfedf7a43c24150099e1e7bd41aacb0444e19f', 'score': 0.04130038991570473}, {'rank': 7, 'paperId': '53d11847de929f1cbbff3dd8f758c214ffd28830', 'score': 0.0411248654127121}, {'rank': 8, 'paperId': '799b24723b55242ca57582a823885325db0f79df', 'score': 0.04081790894269943}, {'rank': 9, 'paperId': 'baae51fb72075c4673ad809133d3599e5eb99eba', 'score': 0.04080697149038315}, {'rank': 1